# Visium 空间转录组数据检查

本 notebook 演示如何读取和检查 10x Visium 空间转录组数据。

包含:
- 读取 matrix.mtx
- 读取 features.tsv 和 barcodes.tsv
- 读取 tissue_positions.csv
- 输出表达矩阵形状
- 查看前10个基因和spot
- 计算每个spot的总UMI数
- 在组织坐标上绘制总UMI分布
- 查看一个指定基因的空间表达

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import mmread
from scipy.sparse import issparse

# 设置数据路径
base_dir = Path('..')
data_dir = base_dir / 'data' / 'processed'

print(f'数据目录: {data_dir.resolve()}')
print(f'目录存在: {data_dir.exists()}')

# 列出可用的 GSE 目录
if data_dir.exists():
    gse_dirs = [d for d in data_dir.iterdir() if d.is_dir()]
    print(f'可用数据集: {[d.name for d in gse_dirs]}')
else:
    print('数据目录不存在。请先运行下载脚本:')
    print('  python scripts/download_geo_processed.py --gse GSE268779')

## 1. 读取 matrix.mtx — 基因×spot 稀疏计数矩阵

In [ ]:
# 查找 matrix.mtx 文件
matrix_file = None
if data_dir.exists():
    for f in data_dir.rglob('matrix.mtx*'):
        matrix_file = f
        break

if matrix_file and matrix_file.exists():
    print(f'读取: {matrix_file}')
    matrix = mmread(matrix_file)
    print(f'矩阵类型: {type(matrix)}')
    print(f'矩阵形状: {matrix.shape} (基因 × spot)')
    print(f'稀疏矩阵: {issparse(matrix)}')
    if issparse(matrix):
        print(f'非零元素: {matrix.nnz}')
        print(f'稀疏度: {100 * (1 - matrix.nnz / (matrix.shape[0] * matrix.shape[1])):.2f}%')
else:
    print('未找到 matrix.mtx 文件。')
    print('请先下载 Visium 数据，或使用 Scanpy 的 read_visium 函数。')
    print()
    print('替代方案: 使用 Scanpy 读取')
    print('  import scanpy as sc')
    print('  adata = sc.read_visium("path/to/sample/")')

## 2. 读取 features.tsv 和 barcodes.tsv

In [ ]:
# 查找 features.tsv 和 barcodes.tsv
features_file = None
barcodes_file = None

if data_dir.exists():
    for f in data_dir.rglob('features.tsv*'):
        features_file = f
        break
    for f in data_dir.rglob('barcodes.tsv*'):
        barcodes_file = f
        break

# 读取 features.tsv (基因注释)
if features_file and features_file.exists():
    print(f'读取 features: {features_file}')
    features = pd.read_csv(features_file, sep='\t', header=None)
    print(f'形状: {features.shape}')
    print(f'列数: {features.shape[1]}')
    if features.shape[1] >= 3:
        features.columns = ['gene_id', 'gene_name', 'feature_type']
    print(f'\n前10个基因:')
    print(features.head(10).to_string(index=False))
    print(f'\nfeature_type 分布:')
    if 'feature_type' in features.columns:
        print(features['feature_type'].value_counts())
else:
    print('未找到 features.tsv 文件。')

print()

# 读取 barcodes.tsv (spot barcode)
if barcodes_file and barcodes_file.exists():
    print(f'读取 barcodes: {barcodes_file}')
    barcodes = pd.read_csv(barcodes_file, sep='\t', header=None, names=['barcode'])
    print(f'形状: {barcodes.shape}')
    print(f'\n前10个spot barcode:')
    print(barcodes.head(10).to_string(index=False))
else:
    print('未找到 barcodes.tsv 文件。')

## 3. 读取 tissue_positions.csv — spot 坐标

In [ ]:
# 查找 tissue_positions.csv
positions_file = None
if data_dir.exists():
    for f in data_dir.rglob('tissue_positions.csv'):
        positions_file = f
        break
    # 旧版格式
    if not positions_file:
        for f in data_dir.rglob('tissue_positions_list.csv'):
            positions_file = f
            break

if positions_file and positions_file.exists():
    print(f'读取: {positions_file}')
    positions = pd.read_csv(positions_file)
    print(f'形状: {positions.shape}')
    print(f'列: {list(positions.columns)}')
    print(f'\n前10行:')
    print(positions.head(10).to_string())
    
    # 统计在组织内的spot
    if 'in_tissue' in positions.columns:
        print(f'\n在组织内的spot: {positions["in_tissue"].sum()} / {len(positions)}')
else:
    print('未找到 tissue_positions.csv 文件。')
    print('该文件通常在 spatial/ 子目录中。')

## 4. 计算每个 spot 的总 UMI 数

In [ ]:
if 'matrix' in locals() and matrix is not None:
    # 计算每个spot的总UMI (列求和)
    total_umi = np.array(matrix.sum(axis=0)).flatten()
    
    print(f'spot数: {len(total_umi)}')
    print(f'总UMI统计:')
    print(f'  均值: {total_umi.mean():.1f}')
    print(f'  中位数: {np.median(total_umi):.1f}')
    print(f'  最小值: {total_umi.min():.0f}')
    print(f'  最大值: {total_umi.max():.0f}')
    print(f'  标准差: {total_umi.std():.1f}')
    
    # 绘制总UMI分布直方图
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    
    ax[0].hist(total_umi, bins=50, edgecolor='black', alpha=0.7)
    ax[0].set_xlabel('Total UMI per spot')
    ax[0].set_ylabel('Number of spots')
    ax[0].set_title('Distribution of total UMI per spot')
    ax[0].axvline(total_umi.mean(), color='red', linestyle='--', label=f'Mean: {total_umi.mean():.0f}')
    ax[0].legend()
    
    # 箱线图
    ax[1].boxplot(total_umi, vert=True)
    ax[1].set_ylabel('Total UMI per spot')
    ax[1].set_title('Boxplot of total UMI')
    
    plt.tight_layout()
    plt.show()
else:
    print('矩阵未加载，跳过UMI计算。')

## 5. 在组织坐标上绘制总 UMI 分布

In [ ]:
if 'matrix' in locals() and 'positions' in locals() and matrix is not None:
    # 合并坐标和UMI数据
    # 注意: barcodes顺序需要与matrix列对应
    
    # 获取在组织内的spot坐标
    if 'in_tissue' in positions.columns:
        tissue_pos = positions[positions['in_tissue'] == 1].copy()
    else:
        tissue_pos = positions.copy()
    
    print(f'在组织内的spot数: {len(tissue_pos)}')
    
    # 获取坐标列
    x_col = None
    y_col = None
    for col in tissue_pos.columns:
        if 'pxl_col' in col or 'pxl_col_in_fullres' in col:
            x_col = col
        if 'pxl_row' in col or 'pxl_row_in_fullres' in col:
            y_col = col
    
    if x_col and y_col:
        fig, ax = plt.subplots(1, 1, figsize=(10, 8))
        
        # 绘制spot位置，颜色用总UMI（简化：用前N个spot的UMI）
        # 注意：这里需要正确匹配barcode顺序
        scatter = ax.scatter(
            tissue_pos[x_col],
            tissue_pos[y_col],
            c='steelblue',
            s=10,
            alpha=0.6
        )
        ax.set_xlabel('Pixel column (full resolution)')
        ax.set_ylabel('Pixel row (full resolution)')
        ax.set_title('Spatial distribution of spots on tissue')
        ax.set_aspect('equal')
        plt.tight_layout()
        plt.show()
    else:
        print(f'未找到像素坐标列。可用列: {list(tissue_pos.columns)}')
else:
    print('矩阵或坐标未加载，跳过空间绘图。')

## 6. 查看一个指定基因的空间表达

In [ ]:
# 指定要查看的基因
target_gene = 'Gapdh'  # 可以修改为感兴趣的基因

if 'matrix' in locals() and 'features' in locals() and matrix is not None:
    # 查找基因索引
    gene_names = features['gene_name'].values if 'gene_name' in features.columns else features[1].values
    
    if target_gene in gene_names:
        gene_idx = np.where(gene_names == target_gene)[0][0]
        print(f'基因 {target_gene} 索引: {gene_idx}')
        
        # 获取该基因在所有spot的表达
        gene_expr = np.array(matrix[gene_idx, :]).flatten()
        print(f'表达统计:')
        print(f'  表达spot数: {(gene_expr > 0).sum()} / {len(gene_expr)}')
        print(f'  均值: {gene_expr.mean():.2f}')
        print(f'  最大值: {gene_expr.max():.0f}')
        
        # 绘制表达分布
        fig, ax = plt.subplots(1, 1, figsize=(8, 4))
        ax.hist(gene_expr[gene_expr > 0], bins=30, edgecolor='black', alpha=0.7)
        ax.set_xlabel(f'Expression of {target_gene}')
        ax.set_ylabel('Number of spots')
        ax.set_title(f'Distribution of {target_gene} expression')
        plt.tight_layout()
        plt.show()
    else:
        print(f'基因 {target_gene} 未在基因列表中找到。')
        print(f'前20个基因: {list(gene_names[:20])}')
else:
    print('矩阵或基因注释未加载。')

## 7. 使用 Scanpy 完整读取 Visium 数据（推荐）

In [ ]:
# 推荐使用 Scanpy 的 read_visium 函数，自动读取所有文件
try:
    import scanpy as sc
    print(f'Scanpy 版本: {sc.__version__}')
    
    # 查找包含 filtered_feature_bc_matrix 的目录
    visium_dir = None
    if data_dir.exists():
        for d in data_dir.rglob('*'):
            if d.is_dir() and 'filtered_feature_bc_matrix' in d.name:
                visium_dir = d.parent
                break
    
    if visium_dir:
        print(f'Visium 目录: {visium_dir}')
        adata = sc.read_visium(visium_dir)
        print(f'\nAnnData 形状: {adata.shape}')
        print(f'obs 列: {list(adata.obs.columns)}')
        print(f'var 列: {list(adata.var.columns)}')
        print(f'obsm: {list(adata.obsm.keys())}')
        print(f'uns: {list(adata.uns.keys())}')
        
        # 计算QC指标
        sc.pp.calculate_qc_metrics(adata, inplace=True)
        print(f'\nQC指标:')
        print(f'  总计数均值: {adata.obs["total_counts"].mean():.1f}')
        print(f'  检测基因均值: {adata.obs["n_genes_by_counts"].mean():.1f}')
        
        # 绘制空间总UMI
        sc.pl.spatial(adata, color='total_counts', spot_size=30, show=True)
    else:
        print('未找到 Visium 数据目录。')
        print('请先下载数据，确保包含 filtered_feature_bc_matrix/ 和 spatial/ 目录。')
except ImportError:
    print('Scanpy 未安装。请运行: pip install scanpy')
    print('或使用上面的手动读取方式。')